In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('data/mnist_train.csv')

In [3]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df = np.array(df)
np.random.shuffle(df)
df = df.T

In [5]:
y_data = df[0]
y_data = y_data.reshape(1, 60000)

x_data = df[1:].T

In [6]:
x_data

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(60000, 784))

In [7]:
y_data

array([[2, 5, 7, ..., 8, 9, 5]], shape=(1, 60000))

Основные функции нейросети 

In [8]:
def init_params():
    W1 = np.random.randn(784, 8) * np.sqrt(2. / 784)
    B1 = np.random.randn(1, 8)
    W2 = np.random.randn(8, 16) * np.sqrt(2. / 8)
    B2 = np.random.randn(1, 16)
    W3 = np.random.randn(16, 16) * np.sqrt(2. / 16)
    B3 = np.random.randn(1, 16)
    W4 = np.random.randn(16, 10) * np.sqrt(2. / 16)
    B4 = np.random.randn(1, 10)
    return W1, B1, W2, B2, W3, B3, W4, B4

def create_batches(y_data, x_data):
    y_batches = y_data.reshape(60, 1000, -1)
    x_batches = x_data.reshape(60, 1000, -1)

    return y_batches, x_batches

def forward(x_batch, weights, bias):
    Z1 = x_batch @ weights[0] + bias[0]
    A1 = tanh(Z1)
    
    Z2 = A1 @ weights[1] + bias[1]
    A2 = tanh(Z2)
    
    Z3 = A2 @ weights[2] + bias[2]
    A3 = tanh(Z3)
    
    Z4 = A3 @ weights[3] + bias[3]
    y_pred = softmax(Z4)

    return y_pred, Z4, A3, Z3, A2, Z2, A1, Z1

def tanh(Z):
    return np.tanh(Z)

def softmax(Z):
    Z_max = np.max(Z, axis=1, keepdims=True)
    Z_shifted = Z - Z_max
    Z_exp = np.exp(Z_shifted)

    prob = Z_exp / np.sum(Z_exp, axis=1, keepdims=True)
    return prob

def to_one_hot(y_batch, num_classes=10):
    y_flat = y_batch.flatten().astype(int)
    y_one_hot = np.eye(num_classes)[y_flat]
    return y_one_hot

def compute_loss(y_pred, y_true):
    m = y_pred.shape[0]
    epsilon = 1e-15
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
    loss_matrix = y_true * np.log(y_pred_clipped)
    cost = -np.sum(loss_matrix) / m
    return cost

def get_accuracy(y_pred, y_true_raw):
    # np.argmax возвращает индекс наибольшего значения в каждой строке
    predictions = np.argmax(y_pred, axis=1)
    
    # Выравниваем реальные метки в одномерный массив (1000,)
    labels = y_true_raw.flatten()
    
    # Сравниваем массивы и считаем процент совпадений
    accuracy = np.mean(predictions == labels)
    return accuracy


Блок для SGD + momentum

In [9]:
def init_momentum_state(weights, bias):
    """Инициализация скоростей (v) нулями для каждого слоя"""
    v_w = [np.zeros_like(w) for w in weights]
    v_b = [np.zeros_like(b) for b in bias]
    return v_w, v_b

def update_sgd_momentum(weights, bias, d_weights, d_bias, v_w, v_b, learning_rate=0.01, beta=0.9):
    """
    Шаг оптимизации SGD + Momentum.
    beta (обычно 0.9) - коэффициент сохранения импульса.
    """
    num_layers = len(weights)
    
    for i in range(num_layers):
        # Обновляем "скорость" для весов и смещений
        v_w[i] = beta * v_w[i] + (1 - beta) * d_weights[i]
        v_b[i] = beta * v_b[i] + (1 - beta) * d_bias[i]
        
        # Обновляем сами параметры
        weights[i] = weights[i] - learning_rate * v_w[i]
        bias[i] = bias[i] - learning_rate * v_b[i]
        
    return weights, bias, v_w, v_b

Блок для RMSProp

In [10]:
def init_rmsprop_state(weights, bias):
    """Инициализация квадратов градиентов (s) нулями"""
    s_w = [np.zeros_like(w) for w in weights]
    s_b = [np.zeros_like(b) for b in bias]
    return s_w, s_b

def update_rmsprop(weights, bias, d_weights, d_bias, s_w, s_b, learning_rate=0.001, beta=0.999, epsilon=1e-8):
    """
    Шаг оптимизации RMSProp.
    epsilon - микрочисло для защиты от деления на ноль.
    """
    num_layers = len(weights)
    
    for i in range(num_layers):
        # Накапливаем квадраты градиентов
        s_w[i] = beta * s_w[i] + (1 - beta) * np.square(d_weights[i])
        s_b[i] = beta * s_b[i] + (1 - beta) * np.square(d_bias[i])
        
        # Обновляем параметры (делим градиент на корень из s)
        weights[i] = weights[i] - learning_rate * (d_weights[i] / (np.sqrt(s_w[i]) + epsilon))
        bias[i] = bias[i] - learning_rate * (d_bias[i] / (np.sqrt(s_b[i]) + epsilon))
        
    return weights, bias, s_w, s_b

Блок для Adam

In [11]:
def init_adam_state(weights, bias):
    """Инициализация v (momentum) и s (rmsprop) нулями"""
    v_w = [np.zeros_like(w) for w in weights]
    v_b = [np.zeros_like(b) for b in bias]
    s_w = [np.zeros_like(w) for w in weights]
    s_b = [np.zeros_like(b) for b in bias]
    return v_w, v_b, s_w, s_b

def update_adam(weights, bias, d_weights, d_bias, v_w, v_b, s_w, s_b, t, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
    """
    Шаг оптимизации Adam.
    t - номер текущей итерации (начинается с 1!), нужен для коррекции смещения.
    """
    num_layers = len(weights)
    
    for i in range(num_layers):
        # 1. Считаем экспоненциально взвешенное среднее градиентов (Momentum)
        v_w[i] = beta1 * v_w[i] + (1 - beta1) * d_weights[i]
        v_b[i] = beta1 * v_b[i] + (1 - beta1) * d_bias[i]
        
        # 2. Считаем экспоненциально взвешенное среднее квадратов (RMSProp)
        s_w[i] = beta2 * s_w[i] + (1 - beta2) * np.square(d_weights[i])
        s_b[i] = beta2 * s_b[i] + (1 - beta2) * np.square(d_bias[i])
        
        # 3. Коррекция смещения (Bias correction)
        # Это нужно, чтобы на первых шагах v и s не были слишком близки к нулю
        v_w_corr = v_w[i] / (1 - np.power(beta1, t))
        v_b_corr = v_b[i] / (1 - np.power(beta1, t))
        s_w_corr = s_w[i] / (1 - np.power(beta2, t))
        s_b_corr = s_b[i] / (1 - np.power(beta2, t))
        
        # 4. Обновление весов
        weights[i] = weights[i] - learning_rate * (v_w_corr / (np.sqrt(s_w_corr) + epsilon))
        bias[i] = bias[i] - learning_rate * (v_b_corr / (np.sqrt(s_b_corr) + epsilon))
        
    return weights, bias, v_w, v_b, s_w, s_b

Блок Backward

In [12]:
def backward(x_batch, y_one_hot, weights, y_pred, A3, A2, A1):
    """
    x_batch: входные данные (1000, 784)
    y_one_hot: правильные ответы (1000, 10)
    weights: список матриц весов [W1, W2, W3, W4]
    y_pred: выход Softmax (1000, 10)
    A1, A2, A3: активации скрытых слоев
    """
    m = x_batch.shape[0] # Размер батча (1000)
    
    # --------------------------------------------------
    # Слой 4 (Выходной: Softmax + Cross-Entropy)
    # --------------------------------------------------
    # Градиент для последнего слоя считается простым вычитанием
    dZ4 = y_pred - y_one_hot
    
    # Градиент весов 4-го слоя: умножаем активации 3-го слоя на dZ4
    dW4 = (1 / m) * np.dot(A3.T, dZ4)
    # Градиент смещений: сумма ошибок по всем примерам батча
    dB4 = (1 / m) * np.sum(dZ4, axis=0, keepdims=True)
    
    # --------------------------------------------------
    # Слой 3 (Скрытый: Tanh)
    # --------------------------------------------------
    # Пробрасываем ошибку назад: умножаем ошибку 4-го слоя на транспонированные веса 4-го слоя
    dA3 = np.dot(dZ4, weights[3].T)
    # Умножаем на производную tanh
    dZ3 = dA3 * (1 - np.power(A3, 2))
    
    dW3 = (1 / m) * np.dot(A2.T, dZ3)
    dB3 = (1 / m) * np.sum(dZ3, axis=0, keepdims=True)
    
    # --------------------------------------------------
    # Слой 2 (Скрытый: Tanh)
    # --------------------------------------------------
    dA2 = np.dot(dZ3, weights[2].T)
    dZ2 = dA2 * (1 - np.power(A2, 2))
    
    dW2 = (1 / m) * np.dot(A1.T, dZ2)
    dB2 = (1 / m) * np.sum(dZ2, axis=0, keepdims=True)
    
    # --------------------------------------------------
    # Слой 1 (Входной: Tanh)
    # --------------------------------------------------
    dA1 = np.dot(dZ2, weights[1].T)
    dZ1 = dA1 * (1 - np.power(A1, 2))
    
    dW1 = (1 / m) * np.dot(x_batch.T, dZ1)
    dB1 = (1 / m) * np.sum(dZ1, axis=0, keepdims=True)
    
    # Упаковываем градиенты в списки, чтобы передать оптимизатору
    d_weights = [dW1, dW2, dW3, dW4]
    d_bias = [dB1, dB2, dB3, dB4]
    
    return d_weights, d_bias

Подготовка и нормализация данных

In [13]:
x_data = x_data / 255.0 
y_batches, x_batches = create_batches(y_data, x_data)
num_batches = x_batches.shape[0]

print("Данные готовы и нормализованы!")

Данные готовы и нормализованы!


Инициализация весов и сброс сети

In [14]:
# Инициализируем веса заново
W1, B1, W2, B2, W3, B3, W4, B4 = init_params()
weights = [W1, W2, W3, W4]
bias = [B1, B2, B3, B4]

# Сбрасываем память оптимизатора Adam
s_w, s_b = init_rmsprop_state(weights, bias)

# Сбрасываем счетчик шагов (ОЧЕНЬ ВАЖНО вынести его сюда!)
#t = 1 

print("Сеть сброшена, веса инициализированы.")

Сеть сброшена, веса инициализированы.


Основной цикл

In [16]:
epochs = 25

for epoch in range(epochs):
    epoch_loss = 0
    epoch_acc = 0

    for b in range(num_batches):
        x_batch = x_batches[b]
        y_batch_raw = y_batches[b]

        y_one_hot = to_one_hot(y_batch_raw)

        y_pred, Z4, A3, Z3, A2, Z2, A1, Z1 = forward(x_batch, weights, bias)

        loss = compute_loss(y_pred, y_one_hot)
        acc = get_accuracy(y_pred, y_batch_raw) # Передаем сырые метки!

        epoch_loss += loss
        epoch_acc += acc

        d_weights, d_bias = backward(x_batch, y_one_hot, weights, y_pred, A3, A2, A1)

        weights, bias, s_w, s_b = update_rmsprop(weights, bias, d_weights, d_bias, s_w, s_b, learning_rate=0.001)

        #t+=1
    avg_loss = epoch_loss / num_batches
    avg_acc = epoch_acc / num_batches
    print(f"Эпоха {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | Точность (Accuracy): {avg_acc*100:.2f}%")

print("Обучение завершено!")

Эпоха 01/25 | Loss: 1.4019 | Точность (Accuracy): 56.01%
Эпоха 02/25 | Loss: 0.6694 | Точность (Accuracy): 83.32%
Эпоха 03/25 | Loss: 0.4954 | Точность (Accuracy): 86.90%
Эпоха 04/25 | Loss: 0.4150 | Точность (Accuracy): 88.84%
Эпоха 05/25 | Loss: 0.3671 | Точность (Accuracy): 90.07%
Эпоха 06/25 | Loss: 0.3366 | Точность (Accuracy): 90.77%
Эпоха 07/25 | Loss: 0.3154 | Точность (Accuracy): 91.32%
Эпоха 08/25 | Loss: 0.2996 | Точность (Accuracy): 91.76%
Эпоха 09/25 | Loss: 0.2872 | Точность (Accuracy): 92.09%
Эпоха 10/25 | Loss: 0.2771 | Точность (Accuracy): 92.33%
Эпоха 11/25 | Loss: 0.2686 | Точность (Accuracy): 92.57%
Эпоха 12/25 | Loss: 0.2612 | Точность (Accuracy): 92.72%
Эпоха 13/25 | Loss: 0.2547 | Точность (Accuracy): 92.88%
Эпоха 14/25 | Loss: 0.2491 | Точность (Accuracy): 93.03%
Эпоха 15/25 | Loss: 0.2441 | Точность (Accuracy): 93.16%
Эпоха 16/25 | Loss: 0.2397 | Точность (Accuracy): 93.29%
Эпоха 17/25 | Loss: 0.2357 | Точность (Accuracy): 93.37%
Эпоха 18/25 | Loss: 0.2320 | То